# Simple RAG

In this notebook you will index a few short passages with **OpenRouter** embeddings, store them in LangChain's **`InMemoryVectorStore`**, then run a small retrieval-augmented generation workflow built with regular Python functions: one function retrieves passages, and another calls the chat model to produce an answer.

In [16]:
from textwrap import dedent

## Environment Setup


### Install dependencies


#### A. on Colab?


In [1]:
%pip install -q langchain langchain_openai langchain_community

/home/halgoz/work/Bootcamp/ai-pros/public/content/W4/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


#### B. Locally?


If using `uv` locally you can `uv sync` (provided you have the `pyproject.toml`).


### Environment variables

The following environment variables are used in this notebook:

| Variable               | Required | Purpose                                  |
|------------------------|----------|------------------------------------------|
| `OPENROUTER_API_KEY`   | Yes      | Authenticates requests to OpenRouter (chat and embeddings) |

#### On Colab?

Store these as Colab **Secrets** (key icon in the left sidebar), using the variable names above (for example `OPENROUTER_API_KEY`). Grant this notebook access to the secrets so they can be read securely at runtime.

#### Locally?

Store these in a `.env` file in your project directory (which should be listed in `.gitignore` so it is never committed). The code below uses `python-dotenv` to load values from this file into environment variables at runtime.



::: {.callout-warning}

API Keys are secrets and thus [**shall never be exposed**](./never_expose_api_keys.qmd).

:::


### Reading environment variables in


In [2]:
import os

In [3]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

## Sample documents

Three short excerpts from a fictional manual:


In [4]:
DOCUMENT1 = """Operating the Climate Control System  Your Googlecar has a climate control system that allows you to adjust the temperature and airflow in the car. To operate the climate control system, use the buttons and knobs located on the center console.  Temperature: The temperature knob controls the temperature inside the car. Turn the knob clockwise to increase the temperature or counterclockwise to decrease the temperature. Airflow: The airflow knob controls the amount of airflow inside the car. Turn the knob clockwise to increase the airflow or counterclockwise to decrease the airflow. Fan speed: The fan speed knob controls the speed of the fan. Turn the knob clockwise to increase the fan speed or counterclockwise to decrease the fan speed. Mode: The mode button allows you to select the desired mode. The available modes are: Auto: The car will automatically adjust the temperature and airflow to maintain a comfortable level. Cool: The car will blow cool air into the car. Heat: The car will blow warm air into the car. Defrost: The car will blow warm air onto the windshield to defrost it."""
DOCUMENT2 = """Your Googlecar has a large touchscreen display that provides access to a variety of features, including navigation, entertainment, and climate control. To use the touchscreen display, simply touch the desired icon.  For example, you can touch the "Navigation" icon to get directions to your destination or touch the "Music" icon to play your favorite songs."""
DOCUMENT3 = """Shifting Gears Your Googlecar has an automatic transmission. To shift gears, simply move the shift lever to the desired position.  Park: This position is used when you are parked. The wheels are locked and the car cannot move. Reverse: This position is used to back up. Neutral: This position is used when you are stopped at a light or in traffic. The car is not in gear and will not move unless you press the gas pedal. Drive: This position is used to drive forward. Low: This position is used for driving in snow or other slippery conditions."""

documents = [DOCUMENT1, DOCUMENT2, DOCUMENT3]

## Indexing

### Embedding Model

* See [embedding models](https://openrouter.ai/models?fmt=cards&output_modalities=embeddings) on OpenRouter.ai
* And [OpenRouter Embeddings API](https://openrouter.ai/docs/api-reference/embeddings) on how to use them

In [7]:
# https://openrouter.ai/docs/api/reference/embeddings
import requests

model_name = "nvidia/llama-nemotron-embed-vl-1b-v2:free"
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

def embed(texts: list[str]):
    response = requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": model_name,
            "input": texts,
        },
    )
    data = response.json()
    return data["data"]

### Vector Store

`InMemoryVectorStore` keeps vectors in process memory—fine for tutorials. Production systems typically use a persisted vector database.

In [ ]:
from langchain_core.embeddings import Embeddings

# Implementation of the Embeddings Interface
# Required for VectorStore (below)
class OpenRouterEmbeddings(Embeddings):
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        # the OpenRouter API expects a list of strings
        response = embed(texts)
        return [r["embedding"] for r in response]

    def embed_query(self, text: str) -> list[float]:
        # embed expects list of 1 string
        response = embed([text])
        return response[0]["embedding"]

embeddings = OpenRouterEmbeddings()

Or using **HuggingFace** to have a model that would run locally (if you have a GPU):

In [ ]:
# from langchain_huggingface import HuggingFaceEmbeddings

# embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-mpnet-base-v2"
# )

In [11]:
from langchain_core.vectorstores import InMemoryVectorStore


vectorstore = InMemoryVectorStore.from_texts(
    texts=documents,
    embedding=embeddings,
    ids=[str(i) for i in range(len(documents))],
)
print(f"Indexed {len(vectorstore.store)} documents")

Indexed 3 documents


In [37]:
query = "How do I adjust the temperature?"

### Retriever

In [ ]:
# Make a retriever for the vector store
retriever = vectorstore.as_retriever()

In [15]:
# Invoke the retriever to get relevant documents
retriever.invoke(query)

[Document(id='0', metadata={}, page_content='Operating the Climate Control System  Your Googlecar has a climate control system that allows you to adjust the temperature and airflow in the car. To operate the climate control system, use the buttons and knobs located on the center console.  Temperature: The temperature knob controls the temperature inside the car. Turn the knob clockwise to increase the temperature or counterclockwise to decrease the temperature. Airflow: The airflow knob controls the amount of airflow inside the car. Turn the knob clockwise to increase the airflow or counterclockwise to decrease the airflow. Fan speed: The fan speed knob controls the speed of the fan. Turn the knob clockwise to increase the fan speed or counterclockwise to decrease the fan speed. Mode: The mode button allows you to select the desired mode. The available modes are: Auto: The car will automatically adjust the temperature and airflow to maintain a comfortable level. Cool: The car will blow

## RAG workflow


A simple query -> retrieval -> augmentation -> generation workflow is shown below:

```{mermaid}
graph LR
    A[Question] --> B[Retriever]
    B --> C[/Relevant docs/]
    C --> D[LLM]
    A --> D
    D --> E[Answer]
```

### A. Select LLM

In [24]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

### Define functions

In [19]:
from langchain.messages import (
    SystemMessage,
    HumanMessage,
)

In [ ]:
def retrieve(query: str):
    docs = retriever.invoke(query)
    return docs


In [ ]:
def generate(query: str, passages: list[str]) -> str:
    # query_oneline = query.replace("\n", " ")
    system_text = dedent("""
        Role: You are a helpful and informative assistant. Answer using the reference passages below.
        Style and Tone: Respond in complete sentences for a non-technical audience in a friendly, conversational tone.
        Note: If a passage is irrelevant, you may ignore it.
    """)
    passages_serialized = "\n".join(f"<passage>{p}</passage>" for p in passages)
    user_text = dedent(f"""
        <question>{query}</question>
        <passages>
        {passages_serialized}
        </passages>
    """)
    msg = llm.invoke([
        SystemMessage(system_text),
        HumanMessage(user_text),
    ])
    return msg.content


### Compose the workflow

In [ ]:
# RAG workflow
def rag_qa_workflow(query: str) -> str:
    passages = retrieve(query)
    answer = generate(query, passages)
    return answer

### Run the workflow


In [29]:
from rich import print

In [33]:
query = "According to the manual, what specific action is required to make the car move when shifted into Neutral?"

result = rag_qa_workflow(query)
print(result)

To get the carmoving while it’s in Neutral, you have to press the gas pedal. The manual explains that in Neutral 
the car won’t move on its own—you need to hit the accelerator to make it go.

In [34]:
query = "Which exact icon on the touchscreen display is used to play songs?"

final_answer = rag_qa_workflow(query)
print(final_answer)

{
    'retrieve_passages': [
        Document(
            id='1',
            metadata={},
            page_content='Your Googlecar has a large touchscreen display that provides access to a variety of 
features, including navigation, entertainment, and climate control. To use the touchscreen display, simply touch 
the desired icon.  For example, you can touch the "Navigation" icon to get directions to your destination or touch 
the "Music" icon to play your favorite songs.'
        ),
        Document(
            id='0',
            metadata={},
            page_content='Operating the Climate Control System  Your Googlecar has a climate control system that 
allows you to adjust the temperature and airflow in the car. To operate the climate control system, use the buttons
and knobs located on the center console.  Temperature: The temperature knob controls the temperature inside the 
car. Turn the knob clockwise to increase the temperature or counterclockwise to decrease the temperature. Airflow: 
The airflow knob controls the amount of airflow inside the car. Turn the knob clockwise to increase the airflow or 
counterclockwise to decrease the airflow. Fan speed: The fan speed knob controls the speed of the fan. Turn the 
knob clockwise to increase the fan speed or counterclockwise to decrease the fan speed. Mode: The mode button 
allows you to select the desired mode. The available modes are: Auto: The car will automatically adjust the 
temperature and airflow to maintain a comfortable level. Cool: The car will blow cool air into the car. Heat: The 
car will blow warm air into the car. Defrost: The car will blow warm air onto the windshield to defrost it.'
        ),
        Document(
            id='2',
            metadata={},
            page_content='Shifting Gears Your Googlecar has an automatic transmission. To shift gears, simply move 
the shift lever to the desired position.  Park: This position is used when you are parked. The wheels are locked 
and the car cannot move. Reverse: This position is used to back up. Neutral: This position is used when you are 
stopped at a light or in traffic. The car is not in gear and will not move unless you press the gas pedal. Drive: 
This position is used to drive forward. Low: This position is used for driving in snow or other slippery 
conditions.'
        )
    ]
}

{'generate_answer': 'The touchscreen shows a **“Music”** icon—just tap that icon to start playing your songs.'}

{'rag_qa_workflow': 'The touchscreen shows a **“Music”** icon—just tap that icon to start playing your songs.'}

In [35]:
from IPython.display import Markdown, display

display(Markdown(final_answer))

The touchscreen shows a **“Music”** icon—just tap that icon to start playing your songs.